# Process data

* Load data and process it to get it to a dataloader
* X
* X

In [1]:
# Predict affinity from antibody and antigen structure
# Indicate which one is antigen, which one is antibody

In [2]:
from __future__ import annotations

import shutil  # Only used to remove processed dir
from pathlib import Path

import pandas as pd
import torch

# This might have an error when coming back to Boston
from graphein.ml.datasets import (
    ProteinGraphDataset,
    ProteinGraphListDataset,
)
from graphein.ml.conversion import GraphFormatConvertor 

from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import (
    add_distance_to_edges,
    add_hydrogen_bond_interactions,
    add_ionic_interactions,
    add_peptide_bonds,
)
from graphein.protein.graphs import construct_graph
from graphein.protein.visualisation import plotly_protein_structure_graph
from torch_geometric.transforms import Compose

from utils.features import (DatasetNodeAnnotator, DatasetEdgeAnnotator)

/home/bobby/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
EDGE_CONSTRUCTION_FNCS = [
    add_peptide_bonds,
    add_ionic_interactions,
    add_hydrogen_bond_interactions,
    add_distance_to_edges,
]

config = ProteinGraphConfig(edge_construction_functions=EDGE_CONSTRUCTION_FNCS)
config.model_dump()

{'granularity': 'CA',
 'keep_hets': [],
 'insertions': True,
 'alt_locs': 'max_occupancy',
 'pdb_dir': None,
 'verbose': False,
 'exclude_waters': True,
 'deprotonate': False,
 'protein_df_processing_functions': None,
 'edge_construction_functions': [<function graphein.protein.edges.distance.add_peptide_bonds(G: 'nx.Graph') -> 'nx.Graph'>,
  <function graphein.protein.edges.distance.add_ionic_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>,
  <function graphein.protein.edges.distance.add_hydrogen_bond_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>,
  <function graphein.protein.edges.distance.add_distance_to_edges(G: 'nx.Graph') -> 'nx.Graph'>],
 'node_metadata_functions': [<function graphein.protein.features.nodes.amino_acid.meiler_embedding(n: str, d: Dict[str, Any], return_array: bool = False) -> Union[pandas.core.series.Series, numpy.ndarray]>],
 'edge_metadata_functions': None,
 'graph_metadata_functions': None,
 'get_contacts_conf

In [4]:
g = construct_graph(config=config, pdb_code="5UM8")
nx_graphs = [g]

/home/bobby/miniconda3/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/bobby/miniconda3/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/bobby/miniconda3/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/bobby/miniconda3/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

/home/bobby/miniconda3/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [5]:
p = plotly_protein_structure_graph(
    g,
    colour_edges_by="kind",
    colour_nodes_by="seq_position",
    label_node_ids=False,
    plot_title="Protein graph with peptide backbone and H-Bonds. \n Nodes coloured by sequence position. Edges coloured by type.",
    node_size_multiplier=1,
)
p.show()

In [6]:
# Set
# used "graph.name" to map labels to structures
g_lab_map = {"5UM8": 5}

In [7]:
convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg")
pyg_graphs = [convertor(g) for g in nx_graphs]

# # Create dataset
# ds = ProteinGraphListDataset(
#     root=".", data_list=graphs, graph_label_map=g_lab_map, name="list_test"
# )
pyg_graphs[0]

Data(edge_index=[2, 3226], node_id=[1498], coords=[1498, 3], name='5UM8', num_nodes=1498)

## Load and process training and validation data!

In [8]:
from pathlib import Path

import pandas as pd
import torch
from graphein.ml import ProteinGraphDataset
from graphein.protein.config import ProteinGraphConfig
from graphein.protein.edges.distance import (
    add_hydrogen_bond_interactions,
    add_ionic_interactions,
    add_peptide_bonds,
)

In [9]:
# Intentionally repeated for an easy copy-paste

EDGE_CONSTRUCTION_FNCS = [
    add_peptide_bonds,
    add_ionic_interactions,
    add_hydrogen_bond_interactions,
    add_distance_to_edges,
]

config = ProteinGraphConfig(edge_construction_functions=EDGE_CONSTRUCTION_FNCS)
config.model_dump()

{'granularity': 'CA',
 'keep_hets': [],
 'insertions': True,
 'alt_locs': 'max_occupancy',
 'pdb_dir': None,
 'verbose': False,
 'exclude_waters': True,
 'deprotonate': False,
 'protein_df_processing_functions': None,
 'edge_construction_functions': [<function graphein.protein.edges.distance.add_peptide_bonds(G: 'nx.Graph') -> 'nx.Graph'>,
  <function graphein.protein.edges.distance.add_ionic_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>,
  <function graphein.protein.edges.distance.add_hydrogen_bond_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>,
  <function graphein.protein.edges.distance.add_distance_to_edges(G: 'nx.Graph') -> 'nx.Graph'>],
 'node_metadata_functions': [<function graphein.protein.features.nodes.amino_acid.meiler_embedding(n: str, d: Dict[str, Any], return_array: bool = False) -> Union[pandas.core.series.Series, numpy.ndarray]>],
 'edge_metadata_functions': None,
 'graph_metadata_functions': None,
 'get_contacts_conf

In [10]:
samplesheet = pd.read_csv("data/dummy_samplesheet.csv")

pdb_paths = samplesheet["structure_path"].tolist()
affinity_values = [torch.tensor(dg) for dg in samplesheet["dg"]]

In [11]:
dataset_convertor = GraphFormatConvertor(
    src_format="nx",
    dst_format="pyg",
    # verbose="all_info",
    columns=[
        "edge_index",
        "coords",
        "node_id",
        "meiler",  # Meiler embeddings
        "kind",  # Type of bond (hydrogen, ionic, etc.)
        "dist_mat",
        "distance",
    ],
)

In [12]:
# def add_x(data):
#     data.x = data.meiler.float()
#     return data

In [13]:
# def add_edge_distance(data):
#     # data.edge_attr = data.distance.reshape(-1, 1)
#     data.edge_attr = data.distance
#     return data

In [14]:
# # Need to manually delete so it's always remade
# processed_dir = Path("processed")
# if processed_dir.exists():
#     shutil.rmtree(processed_dir)

# # Regular ProteinGraphDataset
# ds = ProteinGraphDataset(
#     root=".",
#     paths=pdb_paths,
#     graph_labels=affinity_values,
#     graph_format_convertor=dataset_convertor,
#     graphein_config=config,
#     pre_transform=Compose(
#         [add_x, add_edge_distance]
#     ),  # Adds meiler embeddings as x value and edge distance as edge_attr
# )

In [15]:
node_functions = ["meiler_embedding"]
edge_functions = ["distance_to_edges"]

node_annotator = DatasetNodeAnnotator(node_functions)
edge_annotator = DatasetEdgeAnnotator(edge_functions)

In [16]:
processed_dir = Path("processed")
if processed_dir.exists():
    shutil.rmtree(processed_dir)

# Regular ProteinGraphDataset
ds = ProteinGraphDataset(
    root=".",
    paths=pdb_paths,
    graph_labels=affinity_values,
    graph_format_convertor=dataset_convertor,
    graphein_config=config,
    pre_transform=Compose(
        [node_annotator.set_node_x, edge_annotator.set_edge_attr]
    ),  # Adds meiler embeddings as x value and edge distance as edge_attr
)

Processing...
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.07s/it]
Done!


In [17]:
# class DatasetNodeAnnotator:

#     def _meiler(self, data):
#         return data.meiler.float()

#     def _bar(self):
#         pass

#     def foo(self):
#         class_transformation_fncs = dict(inspect.getmembers(self.__class__, predicate=inspect.isfunction))
#         return class_transformation_fncs
#         # return inspect.getmembers(DatasetNodeAnnotator, predicate=inspect.isfunction)

#     @property
#     def annotator_handler(self) -> dict[str, Callable]:
#         class_transformation_fncs = inspect.getmembers(
#             self.__class__, predicate=inspect.isfunction
#         )
#         handler = {
#             f"{fnc_name.removeprefix("_")}": transform_fnc 
#             for (fnc_name, transform_fnc) in class_transformation_fncs 
#             if fnc_name.startswith("_")
#         }
#         return handler

#     def set_node_x(self, ds: ProteinGraphDataset, features: list[str]) -> None:
#         pass

In [18]:
# from sklearn.preprocessing import MultiLabelBinarizer

In [19]:
# mlb = MultiLabelBinarizer()
# edge_attr = torch.Tensor(mlb.fit_transform(ds[0].kind))
# edge_attr.shape

In [20]:
# TO DO: INPUT GRAPH FEATURES

In [21]:
# Data Handling of Graphs

# A graph is used to model pairwise relations (edges) between objects (nodes).
# A single graph in PyG is described by an instance of torch_geometric.data.Data,
# which holds the following attributes by default:

#     data.x: Node feature matrix with shape [num_nodes, num_node_features]
#     data.edge_index: Graph connectivity in COO format with shape [2, num_edges] and type torch.long
#     data.edge_attr: Edge feature matrix with shape [num_edges, num_edge_features]
#     data.y: Target to train against (may have arbitrary shape), e.g., node-level targets of shape [num_nodes, *] or graph-level targets of shape [1, *]
#     data.pos: Node position matrix with shape [num_nodes, num_dimensions]

# None of these attributes are required. In fact, the Data object is not even restricted
# to these attributes. We can, e.g., extend it by data.face to save the connectivity of
# triangles from a 3D mesh in a tensor with shape [3, num_faces] and type torch.long.

In [22]:
print(ds.num_edge_features)
print(ds.num_features)
print(ds.num_node_features)

1
7
7


In [23]:
ds.print_summary()

ProteinGraphDataset (#graphs=1):
+------------+----------+----------+
|            |   #nodes |   #edges |
|------------+----------+----------|
| mean       |     1498 |     3226 |
| std        |      nan |      nan |
| min        |     1498 |     3226 |
| quantile25 |     1498 |     3226 |
| median     |     1498 |     3226 |
| quantile75 |     1498 |     3226 |
| max        |     1498 |     3226 |
+------------+----------+----------+


/home/bobby/miniconda3/lib/python3.12/site-packages/torch_geometric/data/summary.py:34: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1879.)
  std=data.std().item(),


In [24]:
# num_edge_features is a pyg dataset property (Looks at dataset[0].edge_attr.shape[-1])
# num_node_features is a pyG dataset property (Looks at dataset[0].x.shape[-1])
# num_classes is a pyG dataset property (Looks at len(ds.y))

In [25]:
print(ds.num_classes)
print(ds.num_node_features)
print(ds.num_edge_features)

0
7
1


In [26]:
from torch_geometric.loader import DataLoader

dl = DataLoader(ds, batch_size=32, shuffle=True)
for batch in dl:
    print(batch)
    print("---")

    # Input feaures:
    print(f"edge_index: {batch.edge_index}")  # Self explanatory
    print("")
    print(f"X         : {batch.x}")  # Node features
    print("")
    print(f"edge_attr : {batch.edge_attr}")  # Edge features
    print("")

    # Output value:
    print(f"y         : {batch.graph_y}")
    print("---")

DataBatch(edge_index=[2, 3226], node_id=[1], coords=[1498, 3], meiler=[1498, 7], kind=[1], distance=[3226], dist_mat=[1498, 1498], num_nodes=1498, graph_y=[1], x=[1498, 7], edge_attr=[3226, 1], batch=[1498], ptr=[2])
---
edge_index: tensor([[   0,    1,    1,  ..., 1496, 1496, 1497],
        [   1,    0,    2,  ..., 1495, 1497, 1496]])

X         : tensor([[0.0000, 0.0000, 0.0000,  ..., 6.0700, 0.1300, 0.1500],
        [2.9400, 0.2900, 5.8900,  ..., 5.6700, 0.3000, 0.3800],
        [2.5900, 0.1900, 4.0000,  ..., 6.0400, 0.3900, 0.3100],
        ...,
        [1.2800, 0.0500, 1.0000,  ..., 6.1100, 0.4200, 0.2300],
        [2.6700, 0.0000, 2.7200,  ..., 6.8000, 0.1300, 0.3400],
        [3.0300, 0.1100, 2.6000,  ..., 5.6000, 0.2100, 0.3600]])

edge_attr : tensor([[3.7902],
        [3.7902],
        [3.8245],
        ...,
        [3.8165],
        [3.7876],
        [3.7876]], dtype=torch.float64)

y         : tensor([5])
---


In [27]:
batch.edge_attr.shape

torch.Size([3226, 1])

In [28]:
# from graphein.ml import ProteinGraphListDataset, GraphFormatConvertor
# import graphein.protein as gp

# # Construct graphs
# graphs = gp.construct_graphs_mp(
#     pdb_code_it=["3eiy", "4hhb", "1lds", "2ll6"],
#     return_dict=False
#     )

# # do some transformation
# graphs = [gp.extract_subgraph_from_chains(g, ["A"]) for g in graphs]

# # Convert to PyG Data format
# convertor = GraphFormatConvertor(src_format="nx", dst_format="pyg")
# graphs = [convertor(g) for g in graphs]

# # Create dataset
# ds = ProteinGraphListDataset(root=".", data_list=graphs, name="list_test")

## Load and process new data for prediction!

In [29]:
# copy-paste for easy reimplementation
EDGE_CONSTRUCTION_FNCS = [
    add_peptide_bonds,
    add_ionic_interactions,
    add_hydrogen_bond_interactions,
    add_distance_to_edges,
]

config = ProteinGraphConfig(edge_construction_functions=EDGE_CONSTRUCTION_FNCS)
config.model_dump()

{'granularity': 'CA',
 'keep_hets': [],
 'insertions': True,
 'alt_locs': 'max_occupancy',
 'pdb_dir': None,
 'verbose': False,
 'exclude_waters': True,
 'deprotonate': False,
 'protein_df_processing_functions': None,
 'edge_construction_functions': [<function graphein.protein.edges.distance.add_peptide_bonds(G: 'nx.Graph') -> 'nx.Graph'>,
  <function graphein.protein.edges.distance.add_ionic_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>,
  <function graphein.protein.edges.distance.add_hydrogen_bond_interactions(G: 'nx.Graph', rgroup_df: 'Optional[pd.DataFrame]' = None)>,
  <function graphein.protein.edges.distance.add_distance_to_edges(G: 'nx.Graph') -> 'nx.Graph'>],
 'node_metadata_functions': [<function graphein.protein.features.nodes.amino_acid.meiler_embedding(n: str, d: Dict[str, Any], return_array: bool = False) -> Union[pandas.core.series.Series, numpy.ndarray]>],
 'edge_metadata_functions': None,
 'graph_metadata_functions': None,
 'get_contacts_conf

In [30]:
# # Load structures for prediction!
# ds = ProteinGraphDataset(
#     root=".",
#     paths=["./data/structures/5UM8.pdb"],
#     graphein_config=config,
# )